# QAOA on Max-k-XOR-SAT via Symmetric Tensor Contraction

This notebook reproduces the main results from the paper, demonstrating exact QAOA simulation
on Max-k-XOR-SAT problems using the symmetric tensor contraction algorithm.

**Algorithm**: Computes $\langle Z^{\otimes k} \rangle$ at QAOA depth $p$ on $D$-regular,
$k$-uniform hypergraph trees. Cost $O(p \cdot 4^p)$, independent of $D$, $k$, and the
light-cone size $N_{\text{lc}}$.

**This notebook requires no backend compilation** — it uses precomputed results bundled
with the package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy.optimize import curve_fit

from qokit.max_k_xor_sat import (
    load_precomputed_results,
    load_benchmark_energies,
    get_available_configs,
    light_cone_size,
)

# Publication-quality defaults
plt.rcParams.update({
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 17,
    'legend.fontsize': 11,
    'figure.dpi': 150,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'axes.grid': True,
    'grid.alpha': 0.2,
})

## 1. Load All Precomputed Results

We have optimized QAOA energies for 16 $(k, D)$ configurations at depths $p = 1, \ldots, 16$.

In [ ]:
# Load all available configurations
configs = get_available_configs()
print(f"Available configurations: {len(configs)}")

all_datasets = []
for k, D in configs:
    data = load_precomputed_results(k, D)
    results = data["results"]
    p_vals = sorted(int(p) for p in results)
    energies = [results[str(p)]["objective"] for p in p_vals]
    gammas = [np.array(results[str(p)]["gammas"]) for p in p_vals]
    betas = [np.array(results[str(p)]["betas"]) for p in p_vals]
    all_datasets.append({
        "k": k, "D": D,
        "p": np.array(p_vals),
        "energy": np.array(energies),
        "gammas": gammas,
        "betas": betas,
    })

all_datasets.sort(key=lambda ds: (ds["k"], ds["D"]))

# Load benchmark energies from classical algorithms
bench_data = load_benchmark_energies()
bench_columns = bench_data["columns"]
benchmarks = bench_data["data"]
print(f"Benchmark algorithms: {bench_columns}")
print(f"\nConfigurations:")
for ds in all_datasets:
    print(f"  k={ds['k']}, D={ds['D']}: p=1..{ds['p'][-1]} "
          f"(best energy = {ds['energy'].max():.4f})")

## 2. Full Comparison Table: QAOA vs Classical Algorithms

Satisfaction fraction $s/m$ for all $(k, D)$ configurations at the maximum computed depth.

In [ ]:
# Build comparison table
max_p = max(ds["p"][-1] for ds in all_datasets)

print(f"{'(k,D)':>7} | {'Prange':>8} | {'Sim.Ann.':>8} | {'DQI+BP':>8} | "
      f"{'Regev+FGUM':>10} | {'QAOA(p='+str(max_p)+')':>12}")
print("-" * 72)

for ds in all_datasets:
    key = f"{ds['k']},{ds['D']}"
    bv = benchmarks.get(key, [0, 0, 0, 0])
    qaoa_best = ds["energy"].max()
    best_classical = max(bv)
    marker = "*" if qaoa_best > best_classical else " "
    print(f"({ds['k']},{ds['D']}){' '*(4-len(key))} | "
          f"{bv[0]:>8.4f} | {bv[1]:>8.4f} | {bv[2]:>8.4f} | "
          f"{bv[3]:>10.4f} | {qaoa_best:>11.4f}{marker}")

## 3. All Configurations: Energy vs Depth

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

cmap = plt.cm.tab20
for i, ds in enumerate(all_datasets):
    color = cmap(i / max(len(all_datasets) - 1, 1))
    a = (ds['D'] - 1) * (ds['k'] - 1)
    ax.plot(ds["p"], ds["energy"], "o-", color=color, markersize=3.5,
            linewidth=1.2, label=rf"$k\!=\!{ds['k']},\,D\!=\!{ds['D']}$ ($a\!=\!{a}$)")

ax.set_xlabel(r"QAOA depth $p$")
ax.set_ylabel(r"Satisfaction fraction $s/m$")
ax.set_title("QAOA Performance on Max-k-XOR-SAT")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(fontsize=8, ncol=2, loc="lower right", framealpha=0.9)
fig.tight_layout()
plt.show()

## 4. Deep Dive: $k=3$, $D=5$

Energy curve with power-law fit, comparison to classical benchmarks, and crossover prediction.

In [ ]:
# Select k=3, D=5 dataset
ds = next(d for d in all_datasets if d["k"] == 3 and d["D"] == 5)

# Fit E(p) = E_inf - c / (p + b)^alpha
def energy_model(p, E_inf, c, b, alpha):
    return E_inf - c / (p + b) ** alpha

p0 = [ds["energy"][-1] + 0.05, 0.5, 1.0, 1.0]
bounds = ([ds["energy"][-1], 0, 0, 0.01], [1.0, 50.0, 50.0, 10.0])
popt, pcov = curve_fit(energy_model, ds["p"].astype(float), ds["energy"],
                       p0=p0, bounds=bounds, maxfev=50000)
E_inf, c_fit, b_fit, alpha_fit = popt
print(f"Fit: E_inf = {E_inf:.6f}, c = {c_fit:.4f}, b = {b_fit:.4f}, alpha = {alpha_fit:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(ds["p"], ds["energy"], 'o', color='#1f77b4', markersize=5,
        zorder=5, label="QAOA")

# Fit curve extrapolated
p_fit = np.linspace(ds["p"].min(), ds["p"].max() + 100, 200)
ax.plot(p_fit, energy_model(p_fit, *popt), "-", color="#1f77b4",
        alpha=0.5, linewidth=2)
ax.axhline(E_inf, color="#1f77b4", linestyle=":", alpha=0.5,
           linewidth=2, label=r"QAOA ($p\to\infty$)")

# Benchmarks
bench_key = f"{ds['k']},{ds['D']}"
bench_styles = [
    {"color": "#d62728", "linestyle": "--", "linewidth": 2},
    {"color": "#2ca02c", "linestyle": "--", "linewidth": 2},
    {"color": "#9467bd", "linestyle": "-.", "linewidth": 2},
    {"color": "#ff7f0e", "linestyle": "-.", "linewidth": 2},
]
if bench_key in benchmarks:
    bv = benchmarks[bench_key]
    for i, (name, val) in enumerate(zip(bench_columns, bv)):
        ax.axhline(val, label=name, alpha=0.6, **bench_styles[i])

ax.set_xlabel(r"QAOA depth $p$")
ax.set_ylabel(r"$s/m$")
ax.set_title(rf"Satisfaction fraction vs QAOA depth ($k={ds['k']}$, $D={ds['D']}$)")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(loc='lower right', fontsize=12)
ax.set_xlim(0, ds["p"].max() + 100)
fig.tight_layout()
plt.show()

# Crossover prediction
best_classical = max(benchmarks[bench_key])
if E_inf > best_classical:
    ratio = c_fit / (E_inf - best_classical)
    p_cross = ratio ** (1.0 / alpha_fit) - b_fit
    print(f"\nPredicted crossover depth (QAOA > best classical): p_beat ~ {int(np.ceil(p_cross))}")
else:
    print(f"\nExtrapolation predicts QAOA will not surpass best classical ({best_classical:.4f})")

## 5. Angle Profiles

Optimized $\gamma_i$ and $\beta_i$ plotted against normalized position, colored by depth $p$.

In [ ]:
fig, (ax_g, ax_b) = plt.subplots(1, 2, figsize=(13, 4.5))

cmap = plt.cm.viridis
p_vals = ds["p"]
norm = plt.Normalize(p_vals.min(), p_vals.max())

for p, gammas, betas in zip(p_vals, ds["gammas"], ds["betas"]):
    t = np.linspace(0, 1, p)
    color = cmap(norm(p))
    ax_g.plot(t, gammas, "o-", color=color, markersize=3, linewidth=1)
    ax_b.plot(t, betas, "o-", color=color, markersize=3, linewidth=1)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

for ax, label in [(ax_g, r"$\gamma_i$"), (ax_b, r"$\beta_i$")]:
    ax.set_xlabel(r"Normalized position $i/(p{-}1)$")
    ax.set_ylabel(label)

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(r"Depth $p$")

fig.suptitle(rf"Optimized angle profiles ($k\!=\!{ds['k']},\, D\!=\!{ds['D']}$)",
             fontsize=16)
plt.show()

## 6. Extrapolation Analysis

Fit $E(p) = E_\infty - c/(p+b)^\alpha$ with bootstrap confidence intervals.

In [ ]:
# Bootstrap CI for E_inf
rng = np.random.default_rng(42)
n_boot = 2000
boot_Einfs = []

n = len(ds["p"])
for _ in range(n_boot):
    idx = rng.integers(0, n, size=n)
    p_boot = ds["p"][idx].astype(float)
    e_boot = ds["energy"][idx]
    try:
        bp, _ = curve_fit(energy_model, p_boot, e_boot, p0=popt,
                          bounds=([e_boot.min(), 0, 0, 0.01], [1.0, 50.0, 50.0, 10.0]),
                          maxfev=50000)
        boot_Einfs.append(bp[0])
    except (RuntimeError, ValueError):
        pass

boot_Einfs = np.array(boot_Einfs)
ci_lo, ci_hi = np.percentile(boot_Einfs, [2.5, 97.5])

print(f"k={ds['k']}, D={ds['D']}:")
print(f"  E_inf = {E_inf:.6f}")
print(f"  95% CI = [{ci_lo:.6f}, {ci_hi:.6f}]")
print(f"  Bootstrap std = {boot_Einfs.std():.6f}")
print(f"  Successful fits: {len(boot_Einfs)}/{n_boot}")

# Histogram
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(boot_Einfs, bins=50, density=True, alpha=0.7, color='#1f77b4')
ax.axvline(E_inf, color='red', linewidth=2, label=f'Point estimate: {E_inf:.4f}')
ax.axvline(ci_lo, color='gray', linestyle='--', label=f'95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]')
ax.axvline(ci_hi, color='gray', linestyle='--')
ax.set_xlabel(r'$E_\infty$')
ax.set_ylabel('Density')
ax.set_title(rf'Bootstrap distribution of $E_\infty$ ($k={ds["k"]}, D={ds["D"]}$)')
ax.legend()
fig.tight_layout()
plt.show()

## 7. (Optional) Live Computation

If you have a backend installed (C++ or JAX), you can compute new values:

In [ ]:
from qokit.max_k_xor_sat import contract_symmetric_tree, light_cone_size

# Example: compute <Z^2> for k=2, D=3, p=3 with specific angles
gammas = np.array([0.3, 0.4, 0.5])
betas = np.array([0.4, 0.35, 0.3])

try:
    val = contract_symmetric_tree(gammas, betas, p=3, D=3, k=2)
    N_lc = light_cone_size(3, 3, 2)
    print(f"<Z^2> = {val:.10f}")
    print(f"Energy (satisfaction fraction) = {(1 - val) / 2:.10f}")
    print(f"Light cone size: {N_lc} qubits")
except RuntimeError as e:
    print(f"No backend available: {e}")
    print("To enable live computation, build the C++ backend:")
    print("  cd qokit/max_k_xor_sat/cpp && mkdir -p build && cd build && cmake .. && make -j")
    print("Or install JAX: pip install 'qokit[xorsat-gpu]'")